### Section G: Full User's Profile for upcoming GGSN model

In [1]:
import glob
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
def read_chunks(pattern):
    files = sorted(glob.glob(pattern))
    return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

lf = read_chunks("C:/Users/dagak/test_data/logon_features*.parquet")
print("Login data loaded...")
df = read_chunks("C:/Users/dagak/test_data/device_features*.parquet")
print("Devicde data loaded...")
ef = read_chunks("C:/Users/dagak/test_data/email_features*.parquet")
print("Email data loaded...")

Login data loaded...
Devicde data loaded...
Email data loaded...


In [3]:
ff = read_chunks("C:/Users/dagak/test_data/file_features*.parquet")
print("File data loaded...")

File data loaded...


In [4]:
hf = read_chunks("C:/Users/dagak/test_data/http_features*.parquet")
print("Http data loaded...")

Http data loaded...


In [5]:
print(f"Logon cols: {lf.columns}")

Logon cols: Index(['user', 'year', 'month', 'day', 'login_amount', 'night_logins',
       'different_pcs', 'first_hour', 'last_hour'],
      dtype='object')


In [6]:
print(f"Device cols: {df.columns}")

Device cols: Index(['user', 'year', 'month', 'day', 'usb_connections', 'usb_nightly',
       'computers_usb'],
      dtype='object')


In [7]:
print(f"Email cols: {ef.columns}")

Email cols: Index(['user', 'year', 'month', 'day', 'email amounts', 'external_emails',
       'max_receivers', 'avg_receivers', 'attachments_amount',
       'nightly_emails'],
      dtype='object')


In [8]:
print(f"File cols: {ff.columns}")

File cols: Index(['user', 'year', 'month', 'day', 'amount_of_file_operations',
       'risky_files', 'different_extensions', 'nightly_operations'],
      dtype='object')


In [9]:
print(f"Http cols: {hf.columns}")

Http cols: Index(['user', 'year', 'month', 'day', 'visits_amount', 'different_domains',
       'sus_domains', 'nightly_visits'],
      dtype='object')


In [10]:
key = ["user", "year", "month", "day"]

profil = lf \
    .merge(df, on=key, how="left") \
    .merge(ef, on=key, how="left") \
    .merge(ff, on=key, how="left") \
    .merge(hf, on=key, how="left")

In [14]:
profil = profil.fillna(0)
profil

,user,year,month,day,login_amount,night_logins,different_pcs,first_hour,last_hour,usb_connections,...,attachments_amount,nightly_emails,amount_of_file_operations,risky_files,different_extensions,nightly_operations,visits_amount,different_domains,sus_domains,nightly_visits
0,NPA0366,2010,6,21,7,3,7,0,22,6.0,...,8.0,0.0,13.0,1.0,5.0,10.0,114.0,44.0,0.0,0.0
1,NPA0366,2010,11,17,7,4,7,0,23,7.0,...,11.0,0.0,29.0,4.0,4.0,20.0,114.0,29.0,0.0,0.0
2,DGS0220,2010,3,4,7,5,7,8,22,5.0,...,0.0,0.0,16.0,0.0,2.0,16.0,120.0,43.0,1.0,0.0
3,BJW0369,2010,8,16,7,4,7,8,23,7.0,...,1.0,0.0,27.0,3.0,5.0,4.0,114.0,40.0,0.0,0.0
4,BJW0369,2010,2,11,7,5,7,4,22,8.0,...,4.0,0.0,37.0,1.0,4.0,10.0,114.0,37.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342469,ANC0037,2010,8,24,1,0,1,8,8,7.0,...,15.0,0.0,0.0,0.0,0.0,0.0,143.0,10.0,0.0,0.0
342470,RMH0406,2010,6,25,1,1,1,7,7,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,95.0,23.0,17.0,0.0
342471,JZM0149,2010,7,7,1,0,1,8,8,0.0,...,8.0,0.0,0.0,0.0,0.0,0.0,114.0,22.0,0.0,0.0
342472,KAH0848,2010,7,13,1,0,1,8,8,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,10.0,8.0,0.0,0.0


In [15]:
profil["visits_amount"].nunique()

45

In [16]:
profil["heuristic_risk"] = (
    profil["night_logins"] * 2 +
    profil["usb_nightly"] * 3 +
    profil["external_emails"] * 2 +
    profil["risky_files"] * 2 +
    profil["sus_domains"] * 1
)

In [17]:
profil

,user,year,month,day,login_amount,night_logins,different_pcs,first_hour,last_hour,usb_connections,...,nightly_emails,amount_of_file_operations,risky_files,different_extensions,nightly_operations,visits_amount,different_domains,sus_domains,nightly_visits,heuristic_risk
0,NPA0366,2010,6,21,7,3,7,0,22,6.0,...,0.0,13.0,1.0,5.0,10.0,114.0,44.0,0.0,0.0,27.0
1,NPA0366,2010,11,17,7,4,7,0,23,7.0,...,0.0,29.0,4.0,4.0,20.0,114.0,29.0,0.0,0.0,36.0
2,DGS0220,2010,3,4,7,5,7,8,22,5.0,...,0.0,16.0,0.0,2.0,16.0,120.0,43.0,1.0,0.0,42.0
3,BJW0369,2010,8,16,7,4,7,8,23,7.0,...,0.0,27.0,3.0,5.0,4.0,114.0,40.0,0.0,0.0,34.0
4,BJW0369,2010,2,11,7,5,7,4,22,8.0,...,0.0,37.0,1.0,4.0,10.0,114.0,37.0,0.0,0.0,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342469,ANC0037,2010,8,24,1,0,1,8,8,7.0,...,0.0,0.0,0.0,0.0,0.0,143.0,10.0,0.0,0.0,2.0
342470,RMH0406,2010,6,25,1,1,1,7,7,0.0,...,0.0,0.0,0.0,0.0,0.0,95.0,23.0,17.0,0.0,29.0
342471,JZM0149,2010,7,7,1,0,1,8,8,0.0,...,0.0,0.0,0.0,0.0,0.0,114.0,22.0,0.0,0.0,2.0
342472,KAH0848,2010,7,13,1,0,1,8,8,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,8.0,0.0,0.0,0.0


In [18]:
top20 = profil.sort_values("heuristic_risk", ascending=False)[
    ["user", "year", "month", "day", "heuristic_risk"]
].head(20)

print("Top 20 user-days by heuristic risk:")
print(top20)

Top 20 user-days by heuristic risk:
           user  year  month  day  heuristic_risk
129926  IRG0001  2010      6   16           121.0
219803  IRG0001  2010      8    9           112.0
30534   IEV0003  2010      4   23           104.0
115608  DJL0172  2010      2    1           102.0
41209   IRG0001  2011      1   17           101.0
337779  IRG0001  2010     12    1           101.0
251095  DJL0172  2010     11   23            98.0
226037  IRG0001  2010     10   22            97.0
30841   IRG0001  2011      5    9            96.0
185471  PDF0857  2010     10   28            96.0
299896  IRG0001  2011      4   11            95.0
166625  IRG0001  2010      9   22            94.0
341819  WGB0871  2010      9   20            93.0
142589  IRG0001  2010      3   30            93.0
91690   ECM0004  2010      8    6            93.0
286462  IRG0001  2010     11   22            92.0
311387  IRG0001  2010      6    7            92.0
39237   IRG0001  2010     12   13            90.0
113856  IRG000

In [19]:
profil.count()

user                         342474
year                         342474
month                        342474
day                          342474
login_amount                 342474
night_logins                 342474
different_pcs                342474
first_hour                   342474
last_hour                    342474
usb_connections              342474
usb_nightly                  342474
computers_usb                342474
email amounts                342474
external_emails              342474
max_receivers                342474
avg_receivers                342474
attachments_amount           342474
nightly_emails               342474
amount_of_file_operations    342474
risky_files                  342474
different_extensions         342474
nightly_operations           342474
visits_amount                342474
different_domains            342474
sus_domains                  342474
nightly_visits               342474
heuristic_risk               342474
dtype: int64